In [1]:
import re
import spacy
import pandas as pd
from collections import defaultdict, Counter
import math

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def preprocess(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^\w\s]", " ", text.lower())
    return " ".join(token.lemma_ for token in nlp(text) if not token.is_space)

df = pd.read_csv("articles.csv")

df['title'] = df['title'].fillna("")
df['text'] = df['text'].fillna("")

processed_titles = [preprocess(t) for t in df['title']]
processed_texts = [preprocess(t) for t in df['text']]

def build_inverted_index(processed_docs):
    index = defaultdict(list)
    for doc_id, doc in enumerate(processed_docs):
        words = doc.split()
        unique_words = set(words)
        for word in unique_words:
            index[word].append(doc_id)
    return index

index_title = build_inverted_index(processed_titles)
index_text = build_inverted_index(processed_texts)

def tf_idf(query, inverted_index, processed_docs, M):
    processed_query = preprocess(query)
    query_words = processed_query.split()
    query_tf = Counter(query_words)
    
    scores = defaultdict(float)
    
    for w, c_w_q in query_tf.items():
        if w in inverted_index:
            doc_ids = inverted_index[w]
            df_w = len(doc_ids)
            idf = math.log((M + 1) / df_w)
            
            for doc_id in doc_ids:
                doc_words = processed_docs[doc_id].split()
                c_w_d = doc_words.count(w)
                scores[doc_id] += c_w_q * c_w_d * idf
                
    return scores

def search(query, alpha=0.7):
    M = len(df)
    
    scores_title = tf_idf(query, index_title, processed_titles, M)
    scores_text = tf_idf(query, index_text, processed_texts, M)
    
    combined_scores = {}
    all_doc_ids = set(scores_title.keys()).union(set(scores_text.keys()))
    
    for doc_id in all_doc_ids:
        s_title = scores_title.get(doc_id, 0.0)
        s_text = scores_text.get(doc_id, 0.0)
        combined_scores[doc_id] = alpha * s_title + (1 - alpha) * s_text
        
    sorted_results = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
    
    results_data = []
    for doc_id, score in sorted_results[:10]:
        results_data.append({
            'doc_id': doc_id,
            'title': df['title'].iloc[doc_id],
            'text': df['text'].iloc[doc_id][:100] + "...",
            'scores': score
        })
        
    return pd.DataFrame(results_data)

query = "coursera vs udacity machine learning"
results_df = search(query, alpha=0.7)

display(results_df[['doc_id', 'title', 'scores']])

,doc_id,title,scores
0,276,Coursera vs Udacity for Machine Learning – Hac...,40.478177
1,19,Every single Machine Learning course on the in...,31.463670
2,67,Every single Machine Learning course on the in...,31.463670
3,99,Every single Machine Learning course on the in...,31.463670
4,143,Every single Machine Learning course on the in...,31.463670
5,209,In defense of skepticism about deep learning –...,9.878529
6,327,In defense of skepticism about deep learning –...,9.878529
7,47,Machine Learning in a Week – Learning New Stuf...,8.404892
8,64,Machine Learning in a Year – Learning New Stuf...,7.009145
9,13,Machine Learning is Fun! – Adam Geitgey – Medium,6.956517
